# Campylobacter fully automated scoring and adapted Bayesian negotiations

This notebook converts the earlier multi-profile OPM workflow to the four-party Campylobacter case.

Key design choices:

- Issue/option formulations come from the **public prefix shared by all four supplied prompts**.
- Preliminary discussion comes **only** from `Preliminary_discussion.json/conversation`.
- Step 5 estimates the public-evidence-constrained score distributions without reading participant scores.
- Step 6 creates **one** private prompt combination. For each stakeholder it selects the valid integer draw nearest to the componentwise medians. This retains the 100-point budget and every strict qualitative relation; independently rounding marginal medians would not guarantee either.
- Exactly **one** negotiation is planned. The default moderator is Food Safety Authorities and can be changed below.
- Participant scores are introduced only in the post-hoc diagnostic: interval coverage and equilibrium characterization at the 65 disagreement baseline.
- An integer-only balanced perturbation/threshold grid then asks how little the median-representative landscape must change to obtain a genuine positive-gain NBS, and how the number of strictly IR packages grows with additional adaptation.
- Step 8b repeats the same integer perturbation analysis post hoc on the **original participant scores** (Figs. 6P–7P), with the cap grid extended to 0–10 because that landscape has a 10-point gap at 65.
- Finally, the selected integer-adapted profile is prepared for Bayesian-inference-augmented negotiations under every party as moderator. That final API stage is supplied as code only.

The included cached outputs make the notebook inspectable without API expenditure. Only the optional final negotiation calls a model API.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1 — Mount Drive and locate the bundle

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

BUNDLE_NAME = 'Campy_case_Bayesian'
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'automatic_utility_profiles.py').exists():
    drive_candidate = Path('/content/drive/MyDrive/MicRisk_agents_SDK') / BUNDLE_NAME
    if drive_candidate.exists():
        PROJECT_DIR = drive_candidate
if not (PROJECT_DIR / 'automatic_utility_profiles.py').exists():
    raise FileNotFoundError(
        f'Could not locate {BUNDLE_NAME}. Set PROJECT_DIR to the unzipped bundle folder.'
    )
os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)

def run_step(cmd, show_stdout_chars=2500):
    """Run a pipeline script; on failure show the script's own error instead of a bare CalledProcessError."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout.strip():
        out = result.stdout.strip()
        print(out if len(out) <= show_stdout_chars else '...\n' + out[-show_stdout_chars:])
    if result.returncode != 0:
        print('----- error output -----')
        print(result.stderr[-6000:])
        raise RuntimeError(f'{cmd[1]} failed (exit {result.returncode}); see the error output above.')
    return result


## Step 2 — Install dependencies and configure the optional API key

In [ ]:
%pip -q install -r requirements.txt

# An API key is unnecessary for steps 3–9 when the included cached prior is used.
# Uncomment only before regenerating semantic priors or running the negotiation.
# import getpass
# os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

## Step 3 — Verify the public case inputs

The prepared catalogue was extracted before the exact marker `Your confidential information and preferences:` in every prompt. All four public catalogues matched. The discussion preparation copied only the JSON `conversation` field, removed the moderator opening, removed leading quotations of prior speakers, and removed moderator-note annotations. Issue E's placeholder descriptions are deliberately preserved as supplied.

In [ ]:
catalogue = json.loads(Path('public_info_Campylobacter/public_issue_catalogue.json').read_text())
discussion = json.loads(Path('public_info_Campylobacter/preliminary_discussion.json').read_text())
manifest = json.loads(Path('public_info_Campylobacter/extraction_manifest.json').read_text())

print('Catalogue matches all prompts:', catalogue['cross_prompt_catalogue_match'])
print('Private scores included:', catalogue['private_scores_included'])
print('Discussion source field:', discussion['source_json_field'])
print('Stakeholder message counts:')
for stakeholder, rows in discussion['stakeholders'].items():
    print(f'  {stakeholder}: {len(rows)}')
print('\nIssues and options:')
for issue, row in catalogue['issues'].items():
    print(f"{issue}. {row['title']}")
    for option, option_row in row['options'].items():
        print(f"   {option}: {option_row['title']}")

## Step 4 — Public qualitative priors

The bundle includes a conservative quote-verified cache. Set `REGENERATE_PRIORS=True` to ask a model to independently recode the public discussion. Neither route receives participant scores or the confidential prompt suffixes.

When `REGENERATE_PRIORS=True`, the OpenAI key is loaded here from the Colab secret `api-key`, the original cache is kept as `public_priors_Campylobacter_verified_cache.json`, and Steps 5 and 8 regenerate automatically, because the cached score distributions belong to the old priors. To go back, copy the verified cache over `public_priors_Campylobacter.json` and set the flag to False.

In [ ]:
OPENAI_SECRET_NAME = 'api-key-MicRisk'   # name of the Colab secret holding the OpenAI key
REGENERATE_PRIORS = False   # True = ask the model to re-code the public discussion (API call, ~4 requests)
PRIOR_FILE = Path('public_priors_Campylobacter.json')
VERIFIED_CACHE = Path('public_priors_Campylobacter_verified_cache.json')

if REGENERATE_PRIORS:
    # Needs the OpenAI key now (Step 12 comes later). Uses the Colab secret 'api-key'.
    if not os.environ.get('OPENAI_API_KEY'):
        try:
            from google.colab import userdata
            os.environ['OPENAI_API_KEY'] = userdata.get(OPENAI_SECRET_NAME)
        except Exception:
            import getpass
            os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
    # Keep the quote-verified cache so you can always go back to it.
    if PRIOR_FILE.exists() and not VERIFIED_CACHE.exists():
        VERIFIED_CACHE.write_text(PRIOR_FILE.read_text())
    result = subprocess.run([
        sys.executable, 'generate_discussion_priors.py',
        '--public_info_dir', 'public_info_Campylobacter',
        '--output', str(PRIOR_FILE),
        '--model', 'gpt-5.5', '--force',
    ], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr[-4000:])   # show the real error instead of a bare CalledProcessError
        raise RuntimeError('Prior generation failed; see the error above.')
else:
    print('Using included quote-verified public prior cache.')

priors = json.loads(PRIOR_FILE.read_text())
print('Private scores used:', priors['private_scores_used'])
print('Prior model:', priors.get('model'), '| quotes textually verified:', priors.get('evidence_quotes_textually_verified'))
for stakeholder, row in priors['stakeholders'].items():
    print(stakeholder, '| relevant issues:', row['relevant_issues'],
          '| issue relations:', len(row['issue_relations']),
          '| option relations:', len(row['option_relations']))
if priors.get('validation_warnings'):
    print('Validation warnings:', *priors['validation_warnings'], sep='\n  ')


## Step 5 — Automatic score distributions

For each stakeholder, constrained Monte Carlo produces 20,000 valid integer 100-point tables. The output includes minimum, p05, median, mean, p95, and maximum for every option. These are **model-based uncertainty intervals**, not population confidence intervals.

The same step identifies the closest valid draw to the componentwise median vector. This uses the distribution while avoiding an invalid score table created by independently rounding medians.

In [ ]:
REGENERATE_AUTOMATIC_SCORES = REGENERATE_PRIORS  # new priors -> new score distributions
score_manifest = Path('automatic_scoring/automatic_scoring_manifest.json')
if REGENERATE_AUTOMATIC_SCORES or not score_manifest.exists():
    run_step([
        sys.executable, 'automatic_utility_profiles.py',
        '--prior_file', 'public_priors_Campylobacter.json',
        '--output_dir', 'automatic_scoring',
        '--monte_carlo_samples', '20000',
        '--seed', '20260920',
    ])
else:
    print('Using included 20,000-draw automatic distribution and representative profile.')

ranges = json.loads(Path('automatic_scoring/automatic_score_ranges.json').read_text())
profile = json.loads(Path('automatic_scoring/utility_profiles/profile_median_representative.json').read_text())
print('Samples per stakeholder:', ranges['monte_carlo_samples_per_stakeholder'])
print('Generated profiles:', json.loads(score_manifest.read_text())['generated_profiles'])
for stakeholder, row in profile['stakeholders'].items():
    selection = row['median_representative_selection']
    print(f"{stakeholder}: source draw {row['source_draw_index']}, "
          f"mean |deviation from medians| = {selection['mean_absolute_deviation']}")

## Step 6 — Create one prompt combination and one planned run

One prompt is created for each of the four roles, all using the single median-representative profile. Only the chosen moderator configuration is created, so the experiment plan contains exactly one row rather than a profile × moderator grid.

In [ ]:
MODERATOR = 'Food Safety Authorities'  # change to another exact stakeholder name if desired
REQUIRED_ACCEPTANCES = 2               # matches the supplied Campylobacter prompt protocol

run_step([
    sys.executable, 'prepare_automatic_negotiation_runs.py',
    '--project_dir', '.',
    '--moderator', MODERATOR,
    '--required_acceptances', str(REQUIRED_ACCEPTANCES),
    '--rounds', '12',
    '--base_seed', '1729',
])

plan = json.loads(Path('automatic_prepared_runs/experiment_plan.json').read_text())
assert plan['total_planned_runs'] == 1
print('Planned runs:', plan['total_planned_runs'])
print('Run ID:', plan['runs'][0]['run_id'])
print('Moderator:', plan['runs'][0]['moderator'])
print('Initial deal:', plan['runs'][0]['initial_deal'])

## Step 7 — Fig. 4 equilibrium characterization and participant-score interval check

At disagreement baseline 65, all 1,200 complete packages are enumerated. The report counts individually rational and Pareto-efficient packages and identifies either a Nash-product benchmark or, if the feasible set is empty, the maximin fallback and its gap to 65.

The participant CSV is then compared with the step-5 p05–p95 intervals and full simulated ranges. This is a strictly post-hoc validation; participant scores never feed the automatic distributions or selected profile.

In [ ]:
run_step([sys.executable, 'analyze_scores_and_equilibrium.py'])

equilibrium = json.loads(Path('analysis_outputs/equilibrium_summary.json').read_text())
coverage = json.loads(Path('analysis_outputs/participant_score_interval_coverage_summary.json').read_text())
print(json.dumps(equilibrium, indent=2))
print(json.dumps(coverage['overall'], indent=2))
try:
    from IPython.display import display, Image
    display(Image(filename='analysis_outputs/Fig4_equilibrium_characterization.png'))
    display(Image(filename='analysis_outputs/Fig5_participant_scores_vs_automatic_intervals.png'))
except ImportError:
    print('Figures written to analysis_outputs/.')

## Step 8 — Balanced perturbation and threshold grid

This stage performs bounded **integer** transfers among the median-representative issue weights. Every adapted option score is then re-quantized as an integer; each stakeholder retains a 100-point budget, within-issue rankings/ties/zeros are preserved, and inactive zero-weight issues remain inactive.

The grid evaluates absolute issue-weight caps of 0–5 points and treats each tested threshold as both the acceptance level and disagreement point. It reports two deliberately different results:

1. **Smallest genuine-NBS adaptation:** because scores are integers, require every party to reach at least 66 at a 65 disagreement point; then minimize the largest integer issue-weight change and summed squared changes. This favors several small shifts over one large shift.
2. **IR-rich sampled profile:** maximize the number of strictly IR packages within the ±5-integer-point search space, then prefer smaller perturbations. This measures optionality, but it is not treated as the primary objective because maximizing IR count alone can reward excessive preference distortion.

If the median-representative landscape needs more than ±5 points (e.g. after regenerating priors), the exact search widens the cap one point at a time up to `--max_auto_cap` and extends the grid to include it. If no random sample reaches an NBS, the IR-rich profile falls back to the exact smallest landscape and says so.

In [ ]:
REGENERATE_PERTURBATION = REGENERATE_AUTOMATIC_SCORES
perturbation_report = Path('perturbation_outputs/balanced_perturbation_results.json')
if REGENERATE_PERTURBATION or not perturbation_report.exists():
    run_step([
        sys.executable, 'balanced_perturbation_analysis.py',
        '--profile', 'automatic_scoring/utility_profiles/profile_median_representative.json',
        '--output_dir', 'perturbation_outputs',
        '--caps', '0,1,2,3,4,5',
        '--thresholds', '65,64,63,62,60',
        '--samples_per_cap', '5000',
        '--seed', '1729',
        '--max_auto_cap', '15',   # widens the cap grid automatically if +/-5 cannot produce a genuine NBS
    ])
else:
    print('Using included balanced perturbation and threshold-grid results.')

perturbation = json.loads(perturbation_report.read_text())
smallest = perturbation['smallest_practical_genuine_nbs_profile']
ir_rich = perturbation['ir_rich_profile']
method = perturbation['method']
if method.get('caps_extended_automatically'):
    print(f"NOTE: no genuine NBS within +/-{method['requested_maximum_cap']}; cap grid widened to +/-{method['exact_search_cap_used']}.")
print('Smallest practical genuine NBS:')
print(json.dumps({
    'maximum integer issue-weight change': smallest['adaptation_cost']['maximum_absolute_issue_change'],
    'RMS issue-weight change': smallest['adaptation_cost']['rms_issue_change'],
    'changed stakeholders': smallest['adaptation_cost']['changed_stakeholders'],
    'equilibrium': smallest['equilibrium'],
}, indent=2))
print('\nIR-rich sampled profile (not a certified global optimum):')
print(json.dumps({
    'maximum integer issue-weight change': ir_rich['adaptation_cost']['maximum_absolute_issue_change'],
    'RMS issue-weight change': ir_rich['adaptation_cost']['rms_issue_change'],
    'changed stakeholders': ir_rich['adaptation_cost']['changed_stakeholders'],
    'equilibrium': ir_rich['equilibrium'],
}, indent=2))
try:
    from IPython.display import display, Image
    display(Image(filename='perturbation_outputs/Fig6_balanced_perturbation_grid.png'))
    display(Image(filename='perturbation_outputs/Fig7_selected_adapted_landscapes.png'))
except ImportError:
    print('Perturbation figures written to perturbation_outputs/.')

## Step 8b — Balanced perturbation for the original participant scores (Figs. 6P–7P)

Post-hoc counterpart of Step 8 using the workshop participants' own score tables. The adaptation rules are identical (integer issue-weight transfers, 100-point budget, monotone integer re-quantization preserving rankings/ties/zeros, inactive issues frozen; same smallest-NBS and IR-rich criteria). Because the participant landscape has **no IR package at 65** (maximin 55, gap 10), ±5 points cannot produce an NBS, so the cap grid is extended to 0–10 and the threshold grid down to 54 (where an unadapted strict-IR package first exists). The adapted landscapes are counterfactual and are **not** used to generate prompts.

In [ ]:
REGENERATE_PARTICIPANT_PERTURBATION = False   # ~2 min when True
participant_report = Path('participant_perturbation_outputs/balanced_perturbation_results.json')
if REGENERATE_PARTICIPANT_PERTURBATION or not participant_report.exists():
    run_step([
        sys.executable, 'participant_perturbation_analysis.py',
        '--participant_scores', 'reference_participant_scores/Case1_Campy_Workshop_Alex_final_scores.csv',
        '--output_dir', 'participant_perturbation_outputs',
        '--caps', '0,1,2,3,4,5,6,7,8,9,10',
        '--thresholds', '65,64,63,62,60,58,56,54',
        '--samples_per_cap', '5000',
        '--seed', '1729',
    ])
else:
    print('Using included participant-score perturbation results.')

participant_perturbation = json.loads(participant_report.read_text())
print('Unadapted participant scores at 65:', participant_perturbation['baseline']['benchmark_type'],
      participant_perturbation['baseline']['benchmark_package'],
      participant_perturbation['baseline']['benchmark_utilities'])
for label, key in [('Smallest practical genuine NBS', 'smallest_practical_genuine_nbs_profile'),
                   ('IR-rich sampled profile (not a certified global optimum)', 'ir_rich_profile')]:
    row = participant_perturbation[key]
    print(f'\n{label}:')
    print(json.dumps({
        'maximum integer issue-weight change': row['adaptation_cost']['maximum_absolute_issue_change'],
        'RMS issue-weight change': row['adaptation_cost']['rms_issue_change'],
        'changed stakeholders': row['adaptation_cost']['changed_stakeholders'],
        'equilibrium': row['equilibrium'],
    }, indent=2))
import pandas as pd
print('\nPackages reachable within the cap grid (minimum required issue-weight change):')
print(pd.read_csv('participant_perturbation_outputs/package_minimum_required_issue_change.csv').head(10).to_string(index=False))
try:
    from IPython.display import display, Image
    display(Image(filename='participant_perturbation_outputs/Fig6P_participant_balanced_perturbation_grid.png'))
    display(Image(filename='participant_perturbation_outputs/Fig7P_participant_selected_adapted_landscapes.png'))
except ImportError:
    print('Figures written to participant_perturbation_outputs/.')


## Step 9 — Audit the separation and single-run design

In [ ]:
run_step([sys.executable, 'audit_automatic_scoring_pipeline.py', '--project_dir', '.'])
audit = json.loads(Path('automatic_pipeline_audit.json').read_text())
print(json.dumps(audit, indent=2))
assert audit['status'] == 'passed'

## Step 10 — Materialize the selected integer-adapted profile

The default is the **smallest** integer adaptation that creates a genuine NBS. It is a counterfactual preference landscape, not a claim that participants changed their scores. The materialized profile is valid under the same integer 100-point scoring rule and is the only profile used in the final Bayesian-negotiation stage.

*Fix:* the cached perturbation report previously stored an absolute path from the machine that generated it, so this step failed on Drive. The report now stores project-relative paths, and `create_adapted_integer_profile.py` resolves older reports automatically (or accepts `--source_profile`).

In [ ]:
ADAPTED_PROFILE_SELECTION = 'smallest'  # choose 'ir_rich' only to study the optionality-oriented landscape
ADAPTED_PROFILE = Path(f'adapted_profiles/profile_integer_adapted_{ADAPTED_PROFILE_SELECTION}.json')
run_step([
    sys.executable, 'create_adapted_integer_profile.py',
    '--report', 'perturbation_outputs/balanced_perturbation_results.json',
    '--selection', ADAPTED_PROFILE_SELECTION,
    '--output', str(ADAPTED_PROFILE),
])
adapted = json.loads(ADAPTED_PROFILE.read_text())
for stakeholder, row in adapted['stakeholders'].items():
    assert sum(max(options.values()) for options in row['utility_table'].values()) == 100
    assert all(isinstance(value, int) for options in row['utility_table'].values() for value in options.values())
print('Integer-adapted profile:', ADAPTED_PROFILE)
print('Profile ID:', adapted['profile_id'])

## Step 11 — Preview the Bayesian-inference-augmented, all-moderator design

This reuses the OPM-style setup: the public discussion priors are retained; the live negotiation updates ordinal constraints through `semantic_bayesian_inference.py`; each of the four parties acts as the voting moderator; the moderator opens with its own highest-utility package and produces the final candidate proposal. No participant score table is used.

The defaults preserve the OPM settings: 2 runs per moderator, 12 discussion rounds, history window 6, interim voting interval 6, at least `ADAPTED_REQUIRED_ACCEPTANCES` accepting stakeholders (2 = supplied Campylobacter protocol; 3 = OPM setting), 12,000 posterior samples, model `gpt-5.5`, high reasoning effort, medium verbosity, and seed 1729. This yields 8 prepared API runs and makes no API call.

The private prompts for all eight runs contain the **adapted** score tables selected in Step 10 (by default the smallest adaptation: one integer point for Environmental NGOs, issue B −1 / issue E +1, which re-quantizes B4 23→22, B5 12→11, E1 42→43). The other three stakeholders keep their median-representative tables. The audit re-parses the prompts exactly as the negotiation script does and confirms that they admit the positive-gain NBS.

In [ ]:
ALL_MODERATORS = ','.join(adapted['stakeholders'])
RUNS_PER_MODERATOR = 7
ADAPTED_REQUIRED_ACCEPTANCES = 3
ADAPTED_PREPARED_DIR = Path('adapted_bayesian_prepared_runs')
MAX_TOOL_TURNS = 20   # model/tool steps allowed within one speaker turn

run_step([
    sys.executable, 'prepare_automatic_negotiation_runs.py',
    '--project_dir', '.',
    '--profile_file', str(ADAPTED_PROFILE),
    '--output_dir', str(ADAPTED_PREPARED_DIR),
    '--negotiation_output_root', 'adapted_bayesian_negotiation_outputs',
    '--moderators', ALL_MODERATORS,
    '--runs_per_moderator', str(RUNS_PER_MODERATOR),
    '--required_acceptances', str(ADAPTED_REQUIRED_ACCEPTANCES),
    '--rounds', '12', '--window_size', '6', '--votes_interval', '6',
    '--posterior_samples', '12000',
    '--model', 'gpt-5.5', '--semantic_model', 'gpt-5.5',
    '--reasoning_effort', 'high', '--verbosity', 'medium', '--base_seed', '1729',
    '--max_turns', str(MAX_TOOL_TURNS),
])

adapted_plan = json.loads((ADAPTED_PREPARED_DIR / 'experiment_plan.json').read_text())
assert adapted_plan['total_planned_runs'] == 4 * RUNS_PER_MODERATOR
run_step([
    sys.executable, 'audit_adapted_bayesian_plan.py',
    '--profile', str(ADAPTED_PROFILE),
    '--plan', str(ADAPTED_PREPARED_DIR / 'experiment_plan.json'),
    '--expected_runs_per_moderator', str(RUNS_PER_MODERATOR),
    '--expected_required_acceptances', str(ADAPTED_REQUIRED_ACCEPTANCES),
    '--baseline_profile', 'automatic_scoring/utility_profiles/profile_median_representative.json',
])

# The audit parses every private prompt the 8 runs will load and fails unless those
# score tables equal the adapted profile AND admit a positive-gain NBS at 65.
prompt_check = json.loads((ADAPTED_PREPARED_DIR / 'adapted_bayesian_plan_audit.json').read_text())['final_prompt_check']
print(f"Final prompts checked: {prompt_check['prompts_checked']}")
print('Option scores changed vs. median-representative prompts:')
for change in prompt_check['option_score_changes_vs_baseline']:
    print(f"  {change['stakeholder']}: {change['option']} {change['baseline']} -> {change['final_prompt']} ({change['change']:+d})")
eq = prompt_check['prompt_equilibrium']
print(f"NBS in final prompts: {eq['nbs_package']} {eq['nbs_utilities']} | strict-IR packages: {eq['strict_ir_package_count']}")
print(f"Without adaptation: strict-IR packages = {prompt_check['baseline_equilibrium']['strict_ir_package_count']}")
print('max_turns in run commands:', adapted_plan['runs'][0]['command'][adapted_plan['runs'][0]['command'].index('--max_turns') + 1])
print({key: adapted_plan[key] for key in [
    'total_planned_runs', 'selected_moderators', 'repetitions_per_profile_moderator',
    'rounds', 'window_size', 'votes_interval', 'required_acceptances',
]})

## Step 12 — Optional: configure OpenAI and LangSmith credentials

This cell follows the OPM notebook’s API and EU LangSmith settings. It is intentionally opt-in: retrieve the secrets only if you plan to run the final stage. `OPENAI_API_KEY` is required; LangSmith tracing is optional.

In [ ]:
CONFIGURE_API = False   # set True before Step 13
ENABLE_LANGSMITH_TRACING = True

if CONFIGURE_API:
    try:
        from google.colab import userdata
        openai_key = userdata.get(OPENAI_SECRET_NAME)
        langsmith_key = userdata.get('langsmith') if ENABLE_LANGSMITH_TRACING else None
    except Exception:
        import getpass
        openai_key = getpass.getpass('OpenAI API key: ')
        langsmith_key = getpass.getpass('LangSmith API key (optional): ') if ENABLE_LANGSMITH_TRACING else None
    assert openai_key, 'OpenAI API key was not loaded'
    os.environ['OPENAI_API_KEY'] = openai_key
    if ENABLE_LANGSMITH_TRACING:
        assert langsmith_key, 'LangSmith API key was not loaded'
        os.environ.pop('OPENAI_AGENTS_DISABLE_TRACING', None)
        os.environ['OPENAI_AGENTS_TRACE_INCLUDE_SENSITIVE_DATA'] = 'true'
        os.environ['LANGSMITH_TRACING'] = 'true'
        os.environ['LANGSMITH_API_KEY'] = langsmith_key
        os.environ['LANGSMITH_PROJECT'] = 'campylobacter-integer-adapted-bayesian-negotiation_acceptance_3'
        os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
        os.environ['LANGSMITH_TRACING_SAMPLING_RATE'] = '1.0'
        os.environ.pop('LANGSMITH_HIDE_INPUTS', None)
        os.environ.pop('LANGSMITH_HIDE_OUTPUTS', None)
    print('OpenAI key loaded:', bool(os.environ.get('OPENAI_API_KEY')))

## Step 13 — Optional: run the eight adapted Bayesian negotiations

This is the costly stage. It is disabled by default and uses the prepared all-moderator plan above. `--resume` makes reruns safe by skipping completed output folders. The negotiation is structured as discussion followed by the moderator’s final proposal; it does not add an independent all-party final-vote stage.

Progress is streamed into the notebook. Each run writes `run_summary.json` with `outcome` = `agreement` or `no_agreement`; for no agreement it records the disagreement payoffs (65 each), the rejected final proposal and the main blocking party with its shortfall and costliest issue. A speaker that exhausts its tool steps answers once more without tools instead of aborting the run; a failed run is logged and the next one starts. `--resume` skips finished runs; partial folders of crashed attempts are moved to `_aborted_attempts/`.

In [ ]:
RUN_ADAPTED_BAYESIAN_NEGOTIATIONS = False   # set True to run (costly: RUNS_PER_MODERATOR x 4 negotiations)
if RUN_ADAPTED_BAYESIAN_NEGOTIATIONS:
    assert os.environ.get('OPENAI_API_KEY'), 'Run Step 12 with CONFIGURE_API=True first.'
    command = [sys.executable, '-u', 'run_automatic_profile_experiments.py',
               '--plan', 'adapted_bayesian_prepared_runs/experiment_plan.json', '--resume']
    # command += ['--max_runs', '1']   # uncomment for a single test run first
    if ENABLE_LANGSMITH_TRACING:
        command.append('--trace_sdk_tool_calls_to_langsmith')
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:          # stream progress into the notebook
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError(f'Negotiation runner failed (exit {proc.returncode}); see output above.')
    results = Path('adapted_bayesian_prepared_runs/automatic_experiment_results.csv')
    if results.exists():
        import pandas as pd
        r = pd.read_csv(results)
        print('\nOutcomes by moderator:')
        print(r.groupby('moderator').outcome.value_counts().unstack(fill_value=0))
else:
    print('Negotiations not run. Set CONFIGURE_API=True in Step 12 and RUN_ADAPTED_BAYESIAN_NEGOTIATIONS=True here.')
